In [1]:
import duckdb
import pandas as pd
import re

In [2]:
# Load the source transaction extract.
df = duckdb.read_parquet("../data/raw/transactions.parquet").df()

In [3]:
# Keep records whose notes identify a trade.
query = """
    SELECT * FROM df
    WHERE Notes LIKE '%trade%'
"""

In [4]:
trades = duckdb.sql(query).df().reset_index()

In [5]:
# Identify trades that were voided, rescinded, or nullified, while preserving the reviewed exception.
nulls = """
    SELECT * FROM trades
    WHERE Notes LIKE '%voided%' 
    OR Notes LIKE '%rescinded%' 
    OR Notes LIKE '%nullified%'
    OR (
           Notes LIKE '%Johnson returned%'
           AND Notes LIKE '%previously traded%'
           AND Notes LIKE '%commissioner%'
       )
    OR (
            Acquired IS NULL
            AND Relinquished IS NULL
        )
    """

In [6]:
nullvalues = duckdb.sql(nulls).df()

In [7]:
# Remove the incomplete transaction rows.
clean_trades = trades[~trades["index"].isin(nullvalues["index"])].copy()

In [8]:
clean_trades[clean_trades["Notes"].str.contains("ammended")]

,index,Date,Team,Acquired,Relinquished,Notes,page_url
1881,1881,2001-08-17,Mavericks,None,• future considerations (?),ammended 8/12/01 trade with Rockets,http://prosportstransactions.com/basketball/Se...
1882,1882,2001-08-17,Rockets,• future considerations (?),None,ammended 8/12/01 trade with Mavericks,http://prosportstransactions.com/basketball/Se...


In [9]:
clean_trades[clean_trades["Notes"].isnull()]

,index,Date,Team,Acquired,Relinquished,Notes,page_url


In [10]:
clean_trades[clean_trades["Notes"].str.contains("earlier trade")]

,index,Date,Team,Acquired,Relinquished,Notes,page_url
30,30,1977-04-11,Pacers,None,• Darnell Hillman,sent to Nets as future considerations in earli...,http://prosportstransactions.com/basketball/Se...
31,31,1977-04-11,Nets,• Darnell Hillman,None,received from Pacers as future considerations ...,http://prosportstransactions.com/basketball/Se...
97,97,1977-11-25,Braves,None,• Gus Gerard,sent to Pistons to complete earlier trade invo...,http://prosportstransactions.com/basketball/Se...
98,98,1977-11-25,Pistons,• Gus Gerard,None,received from Braves to complete earlier trade...,http://prosportstransactions.com/basketball/Se...
149,149,1978-08-15,Celtics,None,• Sidney Wicks,sent to Clippers as future considerations in e...,http://prosportstransactions.com/basketball/Se...
150,150,1978-08-15,Clippers,• Sidney Wicks,None,received from Celtics as future considerations...,http://prosportstransactions.com/basketball/Se...
199,199,1979-02-14,Celtics,None,• Tom Barker / Tommy Barker,sent to Knicks to complete earlier trade invol...,http://prosportstransactions.com/basketball/Se...
200,200,1979-02-14,Knicks,• Tom Barker / Tommy Barker,None,received from Celtics to complete earlier trad...,http://prosportstransactions.com/basketball/Se...
203,203,1979-04-12,Pistons,None,• Earl Tatum• cash,sent to Cavaliers to complete earlier trade fo...,http://prosportstransactions.com/basketball/Se...
204,204,1979-04-12,Cavaliers,• Earl Tatum• cash,None,received from Pistons to complete earlier trad...,http://prosportstransactions.com/basketball/Se...


In [11]:
clean_trades[clean_trades["Notes"].str.contains("to complete")]

,index,Date,Team,Acquired,Relinquished,Notes,page_url
97,97,1977-11-25,Braves,None,• Gus Gerard,sent to Pistons to complete earlier trade invo...,http://prosportstransactions.com/basketball/Se...
98,98,1977-11-25,Pistons,• Gus Gerard,None,received from Braves to complete earlier trade...,http://prosportstransactions.com/basketball/Se...
199,199,1979-02-14,Celtics,None,• Tom Barker / Tommy Barker,sent to Knicks to complete earlier trade invol...,http://prosportstransactions.com/basketball/Se...
200,200,1979-02-14,Knicks,• Tom Barker / Tommy Barker,None,received from Celtics to complete earlier trad...,http://prosportstransactions.com/basketball/Se...
203,203,1979-04-12,Pistons,None,• Earl Tatum• cash,sent to Cavaliers to complete earlier trade fo...,http://prosportstransactions.com/basketball/Se...
204,204,1979-04-12,Cavaliers,• Earl Tatum• cash,None,received from Pistons to complete earlier trad...,http://prosportstransactions.com/basketball/Se...
599,599,1983-08-30,Clippers,• Mark Radford,None,received from Sonics to complete earlier trade...,http://prosportstransactions.com/basketball/Se...
600,600,1983-08-30,Sonics,None,• Mark Radford,sent to Clippers to complete earlier trade inv...,http://prosportstransactions.com/basketball/Se...
1527,1527,1997-06-23,Blazers,None,• 1997 first round pick (#20-Paul Grant),sent to Timberwolves to complete earlier trade...,http://prosportstransactions.com/basketball/Se...
1528,1528,1997-06-23,Timberwolves,• 1997 first round pick (#20-Paul Grant),None,received from Blazers to complete earlier trad...,http://prosportstransactions.com/basketball/Se...


In [12]:
clean_trades[clean_trades["Notes"].str.contains("exercised")]

,index,Date,Team,Acquired,Relinquished,Notes,page_url


In [13]:
# Convert bullet-delimited asset text to consistent comma-separated values.
def clean_asset_cell(value):
    if pd.isna(value):
        return value

    assets = re.split(r"\s*•\s*", str(value))

    assets = [re.sub(r"\s+", " ", asset).strip() for asset in assets if asset.strip()]

    return ", ".join(assets)

In [14]:
asset_columns = ["Acquired", "Relinquished"]

clean_trades[asset_columns] = clean_trades[asset_columns].apply(lambda column: column.map(clean_asset_cell))

In [15]:
display(clean_trades[["Acquired", "Relinquished"]].head(20))

,Acquired,Relinquished
0,"1977 first round pick (#15-Brad Davis), cash",Mack Calvin
1,Mack Calvin,"1977 first round pick (#15-Brad Davis), cash"
2,John Mengelt,cash
3,cash,John Mengelt
4,"1977 second round pick (#23-Mike Glenn), cash",Bob Love
5,Bob Love,"1977 second round pick (#23-Mike Glenn), cash"
6,1977 second round pick (#25-Wilson Washington)...,Fred Carter
7,Fred Carter,1977 second round pick (#25-Wilson Washington)...
8,"John Gianelli, cash","Bob McAdoo, Tom McMillen"
9,"Bob McAdoo, Tom McMillen","John Gianelli, cash"


In [16]:
# Verify that no bullet delimiters remain.
for column in asset_columns:
    remaining = clean_trades[column].str.contains("•", na=False).sum()

    print(f"{column}: {remaining} cells still contain bullets")

Acquired: 0 cells still contain bullets
Relinquished: 0 cells still contain bullets


In [17]:
# Save the cleaned historical trade table.
clean_trades.to_csv("../data/interim/cleaned_trades_fixed.csv")